# Gen AI Evaluation サービスを使用した AI エージェントの評価
## 概要

このノートブックでは、Gen AI Evaluation を使用してエージェント評価を行い、ADK（Agent Development Kit）エージェントを評価する方法を学びます。

## 学習目標

このノートブックを完了すると、以下のことができるようになります。
* Gen AI Evaluation サービスで評価するためのローカル ADK エージェントのセットアップ
* エージェント評価データセットの準備
* 単一ツール使用（Single-tool usage）評価のセットアップと使用
* 軌跡（Trajectory）評価の使用
* レスポンス評価の使用
* 独自のデータセット持ち込み（BYOD: Bring-Your-Own-Dataset）セクションを通じて、評価用データセットを提供してエージェントを評価する方法の理解

## 概要

### Gen AI Evaluation サービス
Gen AI Evaluation サービスは、生成 AI モデルおよび AI エージェントの客観的かつデータ駆動型の評価を行うためのエンタープライズグレードのツールを提供します。
モデルの移行、プロンプトの編集、ファインチューニング、AI エージェントの評価など、多くの開発タスクをサポートし、意思決定に役立つ情報を提供します。
ADK には独自の組み込み評価モジュールも用意されていますが、このノートブックでは Agent Platform 生成 AI 評価サービスを使用して ADK ベースのエージェントのパフォーマンスを評価する方法を示します。
このアプローチにより、カスタムメトリクスや人間の判断に沿ったベンチマークを使用して生成モデルやアプリケーションを評価するための、より広範で説明可能かつ品質管理されたツールキットが提供されます。

詳細については、[Gen AI Evaluation サービス](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/evaluation-overview) のドキュメントをご覧ください。

## ライブラリのインポート

In [ ]:
import asyncio
import importlib
import json
import os
import warnings

import pandas as pd
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm  # For multi-model support
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.tool_context import ToolContext
from google.genai import types  # For creating message Content/Parts
from IPython.display import HTML, Markdown, display

# Ignore all warnings
warnings.filterwarnings("ignore")

import logging

logging.basicConfig(level=logging.ERROR)

In [ ]:
GEMINI_LOCATION = "global"
os.environ["GOOGLE_CLOUD_LOCATION"] = GEMINI_LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"  # Use Agent Platform API

In [ ]:
!mkdir -p adk_agents
!mkdir -p adk_agents/agent1_weather_lookup

In [ ]:
%%bash
echo > adk_agents/.env "GOOGLE_CLOUD_LOCATION=$GOOGLE_CLOUD_LOCATION
GOOGLE_GENAI_USE_VERTEXAI=$GOOGLE_GENAI_USE_VERTEXAI
"

## ヘルパー関数の定義

評価結果を可視化するためのプロット用ヘルパー関数のセットを初期化します。
 - 指定された評価メトリクスを含むグループ化された棒グラフ
 - ドリルダウンビューを含む行のサブセットの表示
 - レーダーチャートのプロット

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook"


def plot_bar_plot(
    eval_result: pd.DataFrame, title: str, metrics: list[str] = None
) -> None:
    """pandas DataFrame から指定された評価メトリクスを含むグループ化された棒グラフを表示します。"""
    data = []

    summary_metrics = eval_result.summary_metrics
    if metrics:
        summary_metrics = {
            k: summary_metrics[k]
            for k, v in summary_metrics.items()
            if any(selected_metric in k for selected_metric in metrics)
        }

    data.append(
        go.Bar(
            x=list(summary_metrics.keys()),
            y=list(summary_metrics.values()),
            name=title,
        )
    )

    fig = go.Figure(data=data)

    # バーモードの変更
    fig.update_layout(barmode="group")
    fig.show()


def format_output_as_markdown(output: dict) -> str:
    """出力辞書を整形された Markdown 文字列に変換します。"""
    markdown = "### AI Response\n"
    markdown += f"{output['response']}\n\n"

    if output["predicted_trajectory"]:
        output["predicted_trajectory"] = json.loads(
            output["predicted_trajectory"]
        )
        markdown += "### Function Calls\n"
        for call in output["predicted_trajectory"]:
            markdown += f"- **Function**: `{call['tool_name']}`\n"
            markdown += "  - **Arguments**:\n"
            for key, value in call["tool_input"].items():
                markdown += f"    - `{key}`: `{value}`\n"

    return markdown


def display_dataframe_rows(
    df: pd.DataFrame,
    columns: list[str] | None = None,
    num_rows: int = 3,
    display_drilldown: bool = False,
) -> None:
    """DataFrame の行のサブセットを表示します（オプションでドリルダウンビューを含む）。"""

    if columns:
        df = df[columns]

    base_style = "font-family: monospace; font-size: 14px; white-space: pre-wrap; width: auto; overflow-x: auto;"
    header_style = base_style + "font-weight: bold;"

    for _, row in df.head(num_rows).iterrows():
        for column in df.columns:
            display(
                HTML(
                    f"<span style='{header_style}'>{column.replace('_', ' ').title()}: </span>"
                )
            )
            display(
                HTML(f"<span style='{base_style}'>{row[column]}</span><br>")
            )

        display(HTML("<hr>"))

        if (
            display_drilldown
            and "predicted_trajectory" in df.columns
            and "reference_trajectory" in df.columns
        ):
            display_drilldown(row)


def display_radar_plot(eval_results, title: str, metrics=None):
    """レーダーチャートをプロットします。"""
    fig = go.Figure()
    summary_metrics = eval_results.summary_metrics
    if metrics:
        summary_metrics = {
            k: summary_metrics[k]
            for k, v in summary_metrics.items()
            if any(selected_metric in k for selected_metric in metrics)
        }

    min_val = 0  # = min(summary_metrics.values())
    max_val = max(summary_metrics.values())

    fig.add_trace(
        go.Scatterpolar(
            r=list(summary_metrics.values()),
            theta=list(summary_metrics.keys()),
            fill="toself",
            name=title,
        )
    )
    fig.update_layout(
        title=title,
        polar=dict(radialaxis=dict(visible=True, range=[min_val, max_val])),
        showlegend=True,
    )
    fig.show()

## エージェントファイルの準備

ここでは、[building_agent_with_adk.ipynb](../../building_agents/solutions/building_agent_with_adk.ipynb) ノートブックで作成した基本的なエージェントファイルを再利用します。

まだノートブックを実行していない場合は、以下のセルを実行してファイルを作成してください。

In [ ]:
%%writefile ./adk_agents/agent1_weather_lookup/tools.py
def get_weather(city: str) -> dict:
    """指定された都市の現在の気象レポートを取得します。

    Args:
        city (str): 都市名（例: "New York", "London", "Tokyo"）。

    Returns:
        dict: 気象情報を含む辞書。
              'status' キー（'success' または 'error'）を含みます。
              'success' の場合、気象詳細を含む 'report' キーを含みます。
              'error' の場合、'error_message' キーを含みます。
    """
    print(f"--- Tool: get_weather called for city: {city} ---") # ツールの実行ログ
    city_normalized = city.lower().replace(" ", "") # 基本的な正規化

    # モックの気象データ
    mock_weather_db = {
        "newyork": {"status": "success", "report": "The weather in New York is sunny with a temperature of 25°C."},
        "london": {"status": "success", "report": "It's cloudy in London with a temperature of 15°C."},
        "tokyo": {"status": "success", "report": "Tokyo is experiencing light rain and a temperature of 18°C."},
    }

    if city_normalized in mock_weather_db:
        return mock_weather_db[city_normalized]
    else:
        return {"status": "error", "error_message": f"Sorry, I don't have weather information for '{city}'."}

In [ ]:
%%writefile ./adk_agents/agent1_weather_lookup/agent.py
from google.adk.agents import Agent
MODEL = "gemini-3.8-flash"

from .tools import get_weather

root_agent = Agent(
    name="weather_agent_v1",
    model=MODEL, # Gemini の文字列または LiteLlm オブジェクトを指定可能
    description="Provides weather information for specific cities.",
    instruction="You are a helpful weather assistant. "
                "When the user asks for the weather in a specific city, "
                "use the 'get_weather' tool to find the information. "
                "If the tool returns an error, inform the user politely. "
                "If the tool is successful, present the weather report clearly.",
    tools=[get_weather], # 関数を直接渡す
)

In [ ]:
from adk_agents.agent1_weather_lookup import agent

importlib.reload(agent)  # 強制リロード

# ツールの使用例（オプションのテスト）
print(agent.get_weather("New York"))
print(agent.get_weather("Paris"))

### Runner と SessionService のセットアップ

会話を管理してエージェントを実行するには、さらに 2 つのコンポーネントが必要です。

* `SessionService`: さまざまなユーザーやセッションの会話履歴と状態を管理します。`InMemorySessionService` はすべてをメモリに保存するシンプルな実装であり、テストや単純なアプリケーションに適しています。やり取りされたメッセージを追跡します。状態の永続化についてはステップ 4 で詳しく説明します。
* `Runner`: 対話フローをオーケストレーションするエンジンです。ユーザー入力を受け取り、適切なエージェントにルーティングし、エージェントのロジックに基づいて LLM やツールへの呼び出しを管理し、`SessionService` を介してセッションの更新を処理し、対話の進行状況を表すイベントを生成（yield）します。

まずいくつかの定数を定義します。

In [ ]:
APP_NAME = "weather_info_app"
USER_ID = "user_1"
SESSION_ID = "session_001"  # 簡潔にするため固定 ID を使用

#### Session と Runner の関数の定義

In [ ]:
async def setup_session_and_runner():
    session_service = InMemorySessionService()
    example_session = await session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
    )

    print(f"--- Examining Session Properties ---")
    print(f"ID (`id`):                {example_session.id}")
    print(f"Application Name (`app_name`): {example_session.app_name}")
    print(f"User ID (`user_id`):         {example_session.user_id}")
    print(
        f"State (`state`):           {example_session.state}"
    )  # 注: ここでは初期状態のみを表示
    print(
        f"Events (`events`):         {example_session.events}"
    )  # 初期状態では空
    print(
        f"Last Update (`last_update_time`): {example_session.last_update_time:.2f}"
    )
    print(f"---------------------------------")

    runner = Runner(
        agent=agent.root_agent,
        app_name=APP_NAME,
        session_service=session_service,
    )
    return example_session, runner

### エージェントとのやり取り

エージェントにメッセージを送信し、その応答を受信する方法が必要です。LLM の呼び出しやツールの実行には時間がかかる場合があるため、ADK の `Runner` は非同期で動作します。

以下を行う `async` ヘルパー関数（`call_agent_async`）を定義します。

1. ユーザーのクエリ文字列を受け取る。  
2. それを ADK の `Content` 形式にパッケージ化する。  
3. ユーザー / セッションのコンテキストと新しいメッセージを指定して `runner.run_async` を呼び出す。  
4. ランナーによって生成された **Events** を反復処理する。イベントはエージェントの実行におけるステップ（ツール呼び出しのリクエスト、ツールの結果の受信、LLM の中間思考、最終応答など）を表す。  
5. `event.is_final_response()` を使用して **最終応答（Final Response）** イベントを特定し、出力する。

**なぜ `async` なのか？** LLM や潜在的なツール（外部 API など）とのやり取りは I/O バウンドな操作です。`asyncio` を使用することで、プログラムは実行をブロックすることなくこれらの操作を効率的に処理できます。

### 会話の実行

最後に、エージェントにいくつかのクエリを送信してセットアップをテストします。`async` 呼び出しをメインの `async` 関数でラップし、`await` を使用して実行します。

出力を確認してください。

* ユーザーのクエリが表示されます。  
* エージェントがツールを使用するときに `--- Tool: get_weather called... ---` ログが表示されます。  
* エージェントの最終応答を観察します。天候データが利用できない場合（パリなど）の処理方法も含まれます。

In [ ]:
async def call_agent_async(query):

    print(f"\n>>> User Query: {query}")

    content = types.Content(role="user", parts=[types.Part(text=query)])
    session, runner = await setup_session_and_runner()
    events = runner.run_async(
        user_id=USER_ID, session_id=SESSION_ID, new_message=content
    )

    final_response = ""
    predicted_trajectory_list = []

    async for event in events:
        # アクセスする前に content と parts が存在することを確認
        if not event.content or not event.content.parts:
            continue

        # イベントのコンテンツ内のすべてのパーツを反復処理
        for part in event.content.parts:
            if part.function_call:
                tool_info = {
                    "tool_name": part.function_call.name,
                    "tool_input": dict(part.function_call.args),
                }
                # 何らかの理由で同じ呼び出しが表示された場合に重複を追加しないようにする
                if tool_info not in predicted_trajectory_list:
                    predicted_trajectory_list.append(tool_info)

            # 最終的なテキスト応答は通常、モデルからの最後のイベントに含まれる
            if event.content.role == "model" and part.text:
                # レスポンスを上書き。最後に見つかったテキスト応答が最終的なものである可能性が高い
                final_response = part.text.strip()

        if event.is_final_response():
            final_response = event.content.parts[0].text
            print("Agent Response: ", final_response)

    # 収集された軌跡（trajectory）リストを JSON 文字列にダンプ
    final_output = {
        "response": str(final_response),
        "predicted_trajectory": json.dumps(predicted_trajectory_list),
    }
    return final_output

## Gen AI Evaluation による ADK エージェントの評価

AI エージェントを運用する場合、そのパフォーマンスや動作状況を把握し続けることが重要です。これは主に **モニタリング** と **可観測性（オブザーバビリティ）** の 2 つの観点から捉えることができます。

モニタリングは、エージェントが特定のタスクをどれだけ適切に実行しているかに焦点を当てます。

* **単一ツールの選択（Single Tool Selection）**: エージェントはタスクに適したツールを選択しているか？

* **複数ツールの選択 / 軌跡（Multiple Tool Selection / Trajectory）**: エージェントはツールを使用する順序において論理的な選択を行っているか？

* **レスポンス生成（Response Generation）**: エージェントの出力は優れており、使用したツールに基づいて理にかなっているか？

可観測性は、エージェントの全体的な健全性を理解することに関するものです。

* **レイテンシ（Latency）**: エージェントが応答するまでにどれくらいの時間がかかるか？

* **失敗率（Failure Rate）**: エージェントが応答を生成できない頻度はどの程度か？

Gen AI Evaluation サービスは、エージェントのプロトタイピング中や本番環境へのデプロイ後の両方で、これらすべての側面を評価するのに役立ちます。[事前構築された評価基準とメトリクス](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/determine-eval) を提供するため、エージェントがどのように動作しているかを正確に把握し、改善すべき領域を特定できます。

### エージェント評価用データセットの準備

Gen AI Evaluation サービスを使用して AI エージェントを評価するには、エージェントのどの側面を評価したいかに応じた特定のデータセットが必要です。

このデータセットには、エージェントに与えられるプロンプトを含める必要があります。また、理想的または期待される応答（グラウンドトゥルース/正解）や、エージェントが実行すべき意図されたツール呼び出しシーケンス（参照軌跡: reference trajectory）を含めることもできます。これは、指定された各プロンプトに対してエージェントが呼び出すと期待されるツールのシーケンスを表します。

以下は、ユーザープロンプトと参照軌跡を含むカスタマーサポートエージェント用のデータセットの例です。

In [ ]:
eval_data = {
    "prompt": [
        "Tell me the weather in New York",
        "What is the weather like in London?",
        "How about Paris?",
        "Tell me the weather in New York and London?",
        "What is the weather like in New York and London?",
        "Tell me the weather in New York, London and Paris?",
    ],
    "reference_trajectory": [
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "New York"},
            }
        ],
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "London"},
            }
        ],
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "Paris"},
            }
        ],
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "London"},
            },
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "New York"},
            },
        ],
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "New York"},
            },
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "London"},
            },
        ],
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "London"},
            },
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "Paris"},
            },
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "New York"},
            },
        ],
    ],
}

eval_sample_dataset = pd.DataFrame(eval_data)

データセットからいくつかのサンプルを出力します。

In [ ]:
display(eval_sample_dataset)

In [ ]:
display_dataframe_rows(eval_sample_dataset, num_rows=3)

### 単一ツールの使用（Single Tool Usage）の評価

AI エージェントと評価データセットを設定したら、エージェントが特定のタスクに対して正しい単一のツールを選択しているかどうかの評価を開始します。

#### 単一ツール使用メトリクスの設定

Gen AI Evaluation の `trajectory_single_tool_use` メトリクスを使用すると、特定のツールの順序に関係なく、エージェントが期待するツールを使用しているかどうかを迅速に評価できます。これは、エージェントのプロセスのどこかで適切なツールが使用されたかどうかを評価するための基本的かつ有用な方法です。

`trajectory_single_tool_use` メトリクスを使用するには、特定のユーザーリクエストに対してどのツールが使用されるべきであったかを設定する必要があります。たとえば、ユーザーが「メールを送信して」と依頼した場合、エージェントが "send_email" ツールを使用することを期待し、このメトリクスを使用するときにそのツールの名前を指定します。

In [ ]:
from vertexai.preview.evaluation import EvalTask
from vertexai.preview.evaluation.metrics import (
    PointwiseMetric,
    PointwiseMetricPromptTemplate,
    TrajectorySingleToolUse,
)

In [ ]:
single_tool_usage_metrics = [TrajectorySingleToolUse(tool_name="get_weather")]

#### 評価タスクの実行

評価を実行するには、実験内で事前定義されたデータセット（`eval_sample_dataset`）とメトリクス（この場合は `single_tool_usage_metrics`）を使用して `EvalTask` を開始します。次に、`agent_parsed_outcome` 関数を使用して評価を実行し、この特定の評価実行に一意の識別子を割り当てて、評価結果を保存および可視化します。

In [ ]:
EXPERIMENT_NAME = "evaluate-adk-agent-v1"
PROJECT = !gcloud config get project
PROJECT = PROJECT[0]
BUCKET_NAME = f"agent-evaluation-{PROJECT}-bucket"
BUCKET_URI = f"gs://{BUCKET_NAME}"

**BUCKET の存在を確認し、存在しない場合は作成します:**

In [ ]:
!gcloud storage ls --buckets | grep -w $BUCKET_URI || gcloud storage buckets create --location=us-central1 $BUCKET_URI

In [ ]:
import random
import string
import uuid
from typing import Any


def get_id() -> str:
    """UUID を生成"""
    return str(uuid.uuid4())

ここでは、評価サービスに渡せるように `call_agent_async` を同期関数でラップします。

In [ ]:
def agent_parsed_outcome(query):
    return asyncio.run(call_agent_async(query))

In [ ]:
EXPERIMENT_RUN = f"single-metric-eval-{get_id()}"

single_tool_call_eval_task = EvalTask(
    dataset=eval_sample_dataset,
    metrics=single_tool_usage_metrics,
    experiment=EXPERIMENT_NAME,
    output_uri_prefix=BUCKET_URI + "/single-metric-eval",
)

single_tool_call_eval_result = single_tool_call_eval_task.evaluate(
    runnable=agent_parsed_outcome, experiment_run_name=EXPERIMENT_RUN
)

print(single_tool_call_eval_result)

In [ ]:
def display_eval_report(eval_result: pd.DataFrame) -> None:
    """評価結果を表示します。"""
    metrics_df = pd.DataFrame.from_dict(
        eval_result.summary_metrics, orient="index"
    ).T
    display(Markdown("### Summary Metrics"))
    display(metrics_df)

    display(Markdown("### Row-wise Metrics"))
    display(eval_result.metrics_table)

In [ ]:
display_eval_report(single_tool_call_eval_result)

### 軌跡（Trajectory）の評価

所定のタスクに対してエージェントが最も適切な単一のツールを選択する能力を評価した後、ユーザー入力に対するツール選択シーケンス（軌跡）を分析することで評価を一般化します。これにより、エージェントが適切なツールを選択するだけでなく、それらを合理的かつ効果的な順序で利用しているかどうかが評価されます。

#### 軌跡メトリクスの設定

エージェントの軌跡を評価するために、Gen AI Evaluation はグラウンドトゥルースに基づくいくつかのメトリクスを提供しています。

* `trajectory_exact_match`: 同一の軌跡（同じアクション、同じ順序）

* `trajectory_in_order_match`: 参照アクションが予測された軌跡内に順序どおりに存在する（余分なアクションがあっても許容）

* `trajectory_any_order_match`: すべての参照アクションが予測された軌跡内に存在する（順序や余分なアクションは不問）

* `trajectory_precision`: 予測されたアクションのうち、参照に含まれるものの割合

* `trajectory_recall`: 参照アクションのうち、予測に含まれるものの割合  

すべてのメトリクスは 0 または 1 のスコアですが、`trajectory_precision` と `trajectory_recall` は 0 から 1 の範囲です。

In [ ]:
trajectory_metrics = [
    "trajectory_exact_match",
    "trajectory_in_order_match",
    "trajectory_any_order_match",
    "trajectory_precision",
    "trajectory_recall",
]

#### 評価タスクの実行

新しい `EvalTask` の `evaluate` メソッドを実行して評価を送信します。

In [ ]:
EXPERIMENT_RUN = f"trajectory-{get_id()}"

trajectory_eval_task = EvalTask(
    dataset=eval_sample_dataset,
    metrics=trajectory_metrics,
    experiment=EXPERIMENT_NAME,
    output_uri_prefix=BUCKET_URI + "/multiple-metric-eval",
)

trajectory_eval_result = trajectory_eval_task.evaluate(
    runnable=agent_parsed_outcome, experiment_run_name=EXPERIMENT_RUN
)

In [ ]:
display_eval_report(trajectory_eval_result)

#### 評価結果の可視化

評価結果のサンプルを出力して可視化します。

In [ ]:
display_dataframe_rows(trajectory_eval_result.metrics_table, num_rows=3)

In [ ]:
plot_bar_plot(
    trajectory_eval_result,
    title="Trajectory Metrics",
    metrics=[f"{metric}/mean" for metric in trajectory_metrics],
)

### 最終レスポンスの評価

モデルの評価と同様に、Gen AI Evaluation を使用してエージェントの最終レスポンスを評価できます。

#### レスポンスメトリクスの設定

エージェントの推論後、Gen AI Evaluation は生成されたレスポンスを評価するためのいくつかのメトリクスを提供します。計算ベースのメトリクスを使用してレスポンスを参照と比較したり（必要な場合）、既存またはカスタムのモデルベースのメトリクスを使用して最終レスポンスの品質を判断したりできます。

詳細については、[ドキュメント](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/determine-eval) をご覧ください。

In [ ]:
response_metrics = ["safety", "coherence"]

#### 評価タスクの実行

エージェントの生成されたレスポンスを評価するには、EvalTask クラスの `evaluate` メソッドを使用します。

In [ ]:
EXPERIMENT_RUN = f"response-{get_id()}"

response_eval_task = EvalTask(
    dataset=eval_sample_dataset,
    metrics=response_metrics,
    experiment=EXPERIMENT_NAME,
    output_uri_prefix=BUCKET_URI + "/response-metric-eval",
)

response_eval_result = response_eval_task.evaluate(
    runnable=agent_parsed_outcome, experiment_run_name=EXPERIMENT_RUN
)

display_eval_report(response_eval_result)

#### 評価結果の可視化

新しい評価結果のサンプルを出力します。

In [ ]:
display_dataframe_rows(response_eval_result.metrics_table, num_rows=3)

### ツール選択を条件とした生成レスポンスの評価

環境と相互作用する AI エージェントを評価する場合、一貫性（coherence）のような標準的なテキスト生成メトリクスだけでは不十分な場合があります。これは、これらのメトリクスが主にテキストの構造に焦点を当てているのに対し、エージェントの応答は環境内での有効性に基づいて評価されるべきだからです。

代わりに、このセクションで示すように、エージェントの応答がツールの選択から論理的に導かれているかどうかを評価するカスタムメトリクスを使用します。

#### カスタムメトリクスの定義

[ドキュメント](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/determine-eval#model-based-metrics) によると、基準と評価ルーブリックを設定することで、AI エージェントの応答がそのアクションから論理的に導かれているかどうかを評価するためのプロンプトテンプレートを定義できます。

評価ガイドラインを設定する `criteria` と、スコアリングシステム（1 または 0）を提供する `pointwise_rating_rubric` を定義します。次に、`PointwiseMetricPromptTemplate` を使用して、これらのコンポーネントを用いたテンプレートを作成します。

In [ ]:
criteria = {
    "Follows trajectory": (
        "Evaluate whether the agent's response logically follows from the "
        "sequence of actions it took. Consider these sub-points:\n"
        "  - Does the response reflect the information gathered during the trajectory?\n"
        "  - Is the response consistent with the goals and constraints of the task?\n"
        "  - Are there any unexpected or illogical jumps in reasoning?\n"
        "Provide specific examples from the trajectory and response to support your evaluation."
    )
}

pointwise_rating_rubric = {
    "1": "Follows trajectory",
    "0": "Does not follow trajectory",
}

response_follows_trajectory_prompt_template = PointwiseMetricPromptTemplate(
    criteria=criteria,
    rating_rubric=pointwise_rating_rubric,
    input_variables=["prompt", "predicted_trajectory"],
)

評価で使用できるように、基準とルーブリック情報を組み合わせたこのテンプレートの prompt_data を出力します。

In [ ]:
print(response_follows_trajectory_prompt_template.prompt_data)

評価プロンプトテンプレートを定義したら、レスポンスが特定の軌跡にどれだけ適切に従っているかを評価するための関連メトリクスを設定します。`PointwiseMetric` は、メトリクス名として `response_follows_trajectory` を持ち、先ほど設定した評価用の指示やコンテキストを提供する `response_follows_trajectory_prompt_template` を指定してメトリクスを作成します。

In [ ]:
response_follows_trajectory_metric = PointwiseMetric(
    metric="response_follows_trajectory",
    metric_prompt_template=response_follows_trajectory_prompt_template,
)

#### レスポンスメトリクスの設定

カスタムメトリクスを含めて、新しく生成されたレスポンス評価メトリクスを設定します。

In [ ]:
response_tool_metrics = [
    "trajectory_exact_match",
    "trajectory_in_order_match",
    "safety",
    response_follows_trajectory_metric,
]

#### 評価タスクの実行

エージェントの新しい評価を実行します。

In [ ]:
EXPERIMENT_RUN = f"response-over-tools-{get_id()}"

response_eval_tool_task = EvalTask(
    dataset=eval_sample_dataset,
    metrics=response_tool_metrics,
    experiment=EXPERIMENT_NAME,
    output_uri_prefix=BUCKET_URI + "/reasoning-metric-eval",
)

response_eval_tool_result = response_eval_tool_task.evaluate(
    runnable=agent_parsed_outcome, experiment_run_name=EXPERIMENT_RUN
)

display_eval_report(response_eval_tool_result)

#### 評価結果の可視化

評価結果のサンプルを可視化します。

In [ ]:
display_dataframe_rows(response_eval_tool_result.metrics_table, num_rows=3)

In [ ]:
plot_bar_plot(
    response_eval_tool_result,
    title="Response Metrics",
    metrics=[f"{metric}/mean" for metric in response_tool_metrics],
)

## ボーナス: 独自のデータセット（BYOD）を持ち込んで Gen AI Evaluation で ADK エージェントを評価する

独自のデータセット持ち込み（BYOD: Bring Your Own Dataset）[シナリオ](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/evaluation-dataset) では、予測された軌跡とエージェントからの生成レスポンスの両方を提供します。

### 独自の評価データセットの準備

予測された軌跡と生成されたレスポンスを含む評価データセットを定義します。

In [ ]:
byod_eval_data = {
    "prompt": [
        "Tell me the weather in New York",
        "What is the weather like in London?",
        "How about Paris?",
        "Tell me the weather in New York and London?",
        "What is the weather like in New York and London?",
        "Tell me the weather in New York, London and Paris?",
    ],
    "reference_trajectory": [
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "New York"},
            }
        ],
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "London"},
            }
        ],
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "Paris"},
            }
        ],
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "London"},
            },
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "New York"},
            },
        ],
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "New York"},
            },
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "London"},
            },
        ],
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "London"},
            },
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "Paris"},
            },
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "New York"},
            },
        ],
    ],
    "predicted_trajectory": [
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "New York"},
            }
        ],
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "London"},
            }
        ],
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "Paris"},
            }
        ],
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "London"},
            },
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "New York"},
            },
        ],
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "New York"},
            },
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "London"},
            },
        ],
        [
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "London"},
            },
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "Paris"},
            },
            {
                "tool_name": "get_weather",
                "tool_input": {"city": "New York"},
            },
        ],
    ],
    "response": [
        "The weather in New York is sunny with a temperature of 25°C.",
        "The weather in London is sunny with a temperature of 25°C.",
        "The weather in New York is sunny with a temperature of 25°C.",
        "The weather in New York is sunny with a temperature of 25°C.",
        "The weather in New York is sunny with a temperature of 25°C.",
        "The weather in New York is sunny with a temperature of 25°C.",
    ],
}

byod_eval_sample_dataset = pd.DataFrame(byod_eval_data)
byod_eval_sample_dataset["predicted_trajectory"] = byod_eval_sample_dataset[
    "predicted_trajectory"
].apply(json.dumps)
byod_eval_sample_dataset["reference_trajectory"] = byod_eval_sample_dataset[
    "reference_trajectory"
].apply(json.dumps)

### 評価タスクの実行

独自のデータセットと直前の評価と同じ設定を使用して、新しいエージェント評価を実行します。

In [ ]:
EXPERIMENT_RUN_NAME = f"response-over-tools-byod-{get_id()}"

byod_response_eval_tool_task = EvalTask(
    dataset=byod_eval_sample_dataset,
    metrics=response_tool_metrics,
    experiment=EXPERIMENT_NAME,
    output_uri_prefix=BUCKET_URI + "/byod-eval",
)

byod_response_eval_tool_result = byod_response_eval_tool_task.evaluate(
    experiment_run_name=EXPERIMENT_RUN_NAME
)

display_eval_report(byod_response_eval_tool_result)

### 評価結果の可視化

評価結果のサンプルを可視化します。

In [ ]:
display_dataframe_rows(byod_response_eval_tool_result.metrics_table, num_rows=3)

In [ ]:
display_radar_plot(
    byod_response_eval_tool_result,
    title="ADK agent evaluation",
    metrics=[f"{metric}/mean" for metric in response_tool_metrics],
)

Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.